# Self-Attention Implementation from Scratch

This notebook implements self-attention and multi-head attention, the core mechanisms of Transformers.

**Learning Goals:**
1. Implement scaled dot-product attention
2. Build multi-head attention
3. Add positional encoding
4. Create a complete Transformer block
5. Visualize attention patterns
6. Test on a simple task

## 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Scaled Dot-Product Attention

The fundamental building block:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    
    Args:
        Q: Queries [batch, seq_len, d_k]
        K: Keys [batch, seq_len, d_k]
        V: Values [batch, seq_len, d_v]
        mask: Optional mask [batch, seq_len, seq_len] or [seq_len, seq_len]
    
    Returns:
        output: [batch, seq_len, d_v]
        attention_weights: [batch, seq_len, seq_len]
    """
    d_k = Q.size(-1)
    
    # Compute attention scores
    # [batch, seq_len, d_k] @ [batch, d_k, seq_len] -> [batch, seq_len, seq_len]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Apply mask (optional) - set masked positions to large negative value
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=-1)
    
    # Compute weighted sum of values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

### Test Scaled Dot-Product Attention

In [ ]:
# Simple test
batch_size = 2
seq_len = 4
d_k = 8

Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nAttention weights sum (should be 1.0): {attn_weights[0].sum(dim=-1)}")

## 3. Positional Encoding

Self-attention is permutation invariant, so we need to add positional information.

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the div_term for the sinusoidal functions
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                            -(math.log(10000.0) / d_model))
        
        # Apply sin to even indices
        pe[:, 0::2] = torch.sin(position * div_term)
        # Apply cos to odd indices
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension: [1, max_seq_len, d_model]
        pe = pe.unsqueeze(0)
        
        # Register as buffer (not a parameter, but part of state)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """
        Args:
            x: [batch, seq_len, d_model]
        """
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)

### Visualize Positional Encoding

In [ ]:
# Create and visualize positional encoding
d_model = 128
max_len = 100

pe = PositionalEncoding(d_model, max_len, dropout=0.0)
pe_matrix = pe.pe.squeeze(0).numpy()

plt.figure(figsize=(12, 6))
plt.imshow(pe_matrix.T, cmap='RdBu', aspect='auto')
plt.colorbar()
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Positional Encoding Heatmap')
plt.tight_layout()
plt.show()

# Plot a few dimensions
plt.figure(figsize=(12, 4))
for i in [0, 1, 2, 3, 10, 20, 40, 60]:
    plt.plot(pe_matrix[:50, i], label=f'dim {i}')
plt.xlabel('Position')
plt.ylabel('Value')
plt.title('Positional Encoding - Selected Dimensions')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Multi-Head Attention

Instead of one attention head, we use multiple heads in parallel to learn different types of relationships.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        
        # Output projection
        self.W_O = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def split_heads(self, x):
        """
        Split the last dimension into (num_heads, d_k)
        
        Args:
            x: [batch, seq_len, d_model]
        Returns:
            [batch, num_heads, seq_len, d_k]
        """
        batch_size, seq_len, d_model = x.size()
        
        # Reshape to [batch, seq_len, num_heads, d_k]
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        
        # Transpose to [batch, num_heads, seq_len, d_k]
        return x.transpose(1, 2)
    
    def combine_heads(self, x):
        """
        Combine heads back to original shape
        
        Args:
            x: [batch, num_heads, seq_len, d_k]
        Returns:
            [batch, seq_len, d_model]
        """
        batch_size, num_heads, seq_len, d_k = x.size()
        
        # Transpose to [batch, seq_len, num_heads, d_k]
        x = x.transpose(1, 2)
        
        # Reshape to [batch, seq_len, d_model]
        return x.contiguous().view(batch_size, seq_len, self.d_model)
    
    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q: [batch, seq_len, d_model]
            K: [batch, seq_len, d_model]
            V: [batch, seq_len, d_model]
            mask: Optional [batch, seq_len, seq_len]
        """
        batch_size = Q.size(0)
        
        # 1. Linear projections
        Q = self.W_Q(Q)  # [batch, seq_len, d_model]
        K = self.W_K(K)
        V = self.W_V(V)
        
        # 2. Split into multiple heads
        Q = self.split_heads(Q)  # [batch, num_heads, seq_len, d_k]
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # 3. Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # scores: [batch, num_heads, seq_len, seq_len]
        
        # 4. Apply mask (optional)
        if mask is not None:
            # Expand mask for heads: [batch, 1, seq_len, seq_len]
            mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # 5. Softmax
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # 6. Weighted sum of values
        output = torch.matmul(attention_weights, V)
        # output: [batch, num_heads, seq_len, d_k]
        
        # 7. Combine heads
        output = self.combine_heads(output)  # [batch, seq_len, d_model]
        
        # 8. Final linear projection
        output = self.W_O(output)
        
        return output, attention_weights

### Test Multi-Head Attention

In [ ]:
# Test multi-head attention
d_model = 512
num_heads = 8
batch_size = 2
seq_len = 10

mha = MultiHeadAttention(d_model, num_heads)

x = torch.randn(batch_size, seq_len, d_model)

# Self-attention: Q, K, V all from same source
output, attn_weights = mha(x, x, x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nNumber of parameters: {sum(p.numel() for p in mha.parameters())}")

## 5. Feed-Forward Network

Each Transformer block has a position-wise feed-forward network:

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        # [batch, seq_len, d_model] -> [batch, seq_len, d_ff] -> [batch, seq_len, d_model]
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

## 6. Complete Transformer Block

Combines multi-head attention and feed-forward network with residual connections and layer normalization.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # Multi-head self-attention
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        
        # Feed-forward network
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: [batch, seq_len, d_model]
            mask: Optional [batch, seq_len, seq_len]
        """
        # Self-attention with residual connection and layer norm
        attn_output, attn_weights = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_output))
        
        # Feed-forward with residual connection and layer norm
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_output))
        
        return x, attn_weights

### Test Transformer Block

In [ ]:
d_model = 512
num_heads = 8
d_ff = 2048
batch_size = 2
seq_len = 10

transformer_block = TransformerBlock(d_model, num_heads, d_ff)

x = torch.randn(batch_size, seq_len, d_model)
output, attn_weights = transformer_block(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nNumber of parameters: {sum(p.numel() for p in transformer_block.parameters()):,}")

## 7. Masking for Causal Attention

For language modeling (GPT-style), we need to prevent positions from attending to future positions.

In [ ]:
def create_causal_mask(seq_len):
    """
    Create a causal mask to prevent attention to future positions.
    
    Args:
        seq_len: sequence length
    
    Returns:
        mask: [seq_len, seq_len] lower triangular matrix
    """
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

# Visualize causal mask
mask = create_causal_mask(8)

plt.figure(figsize=(6, 6))
plt.imshow(mask, cmap='Blues')
plt.colorbar()
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Causal Mask (1=allowed, 0=masked)')
plt.tight_layout()
plt.show()

print("Causal mask:")
print(mask.int())
print("\nPosition i can only attend to positions 0...i")

## 8. Simple Sentiment Analysis Example

Let's build a simple sentiment classifier using self-attention.

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, num_classes, 
                 max_seq_len=512, dropout=0.1):
        super().__init__()
        
        # Embedding layers
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        
        # Transformer blocks
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_model * 4, dropout)
            for _ in range(num_layers)
        ])
        
        # Classification head
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, num_classes)
        
        self.d_model = d_model
    
    def forward(self, x, mask=None):
        """
        Args:
            x: [batch, seq_len] - token indices
            mask: Optional attention mask
        """
        # Embed and add positional encoding
        x = self.embedding(x) * math.sqrt(self.d_model)  # Scale embeddings
        x = self.pos_encoding(x)
        
        # Pass through transformer blocks
        attention_weights_list = []
        for transformer_block in self.transformer_blocks:
            x, attn_weights = transformer_block(x, mask)
            attention_weights_list.append(attn_weights)
        
        # Global average pooling
        x = x.mean(dim=1)  # [batch, d_model]
        
        # Classification
        x = self.dropout(x)
        logits = self.fc(x)  # [batch, num_classes]
        
        return logits, attention_weights_list

### Create Dummy Dataset

In [ ]:
# Simple vocabulary
vocab = {
    '<PAD>': 0, '<UNK>': 1,
    'great': 2, 'good': 3, 'excellent': 4, 'amazing': 5, 'love': 6,
    'bad': 7, 'terrible': 8, 'awful': 9, 'hate': 10, 'worst': 11,
    'movie': 12, 'film': 13, 'this': 14, 'is': 15, 'the': 16,
    'a': 17, 'very': 18, 'not': 19, '.': 20
}

# Create simple dataset
# Format: (sentence, label) where label: 0=negative, 1=positive
sentences = [
    ("this movie is great .", 1),
    ("this film is excellent .", 1),
    ("love this movie .", 1),
    ("this is a very good film .", 1),
    ("amazing movie .", 1),
    ("this movie is terrible .", 0),
    ("this film is awful .", 0),
    ("hate this movie .", 0),
    ("this is the worst film .", 0),
    ("bad movie .", 0),
]

def tokenize(sentence, vocab):
    """Convert sentence to token indices"""
    return [vocab.get(word, vocab['<UNK>']) for word in sentence.split()]

def pad_sequence(sequences, max_len, pad_value=0):
    """Pad sequences to same length"""
    padded = []
    for seq in sequences:
        if len(seq) < max_len:
            seq = seq + [pad_value] * (max_len - len(seq))
        else:
            seq = seq[:max_len]
        padded.append(seq)
    return padded

# Prepare data
max_len = 10
X = [tokenize(sent, vocab) for sent, _ in sentences]
y = [label for _, label in sentences]

X = pad_sequence(X, max_len, vocab['<PAD>'])

X_train = torch.tensor(X)
y_train = torch.tensor(y)

print(f"Dataset size: {len(X_train)}")
print(f"Example input: {X_train[0]}")
print(f"Example label: {y_train[0]}")

### Train the Model

In [ ]:
# Model parameters
vocab_size = len(vocab)
d_model = 64
num_heads = 4
num_layers = 2
num_classes = 2

# Create model
model = SentimentClassifier(
    vocab_size=vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    num_classes=num_classes,
    dropout=0.1
).to(device)

print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 100
X_train = X_train.to(device)
y_train = y_train.to(device)

losses = []

model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    # Forward pass
    logits, _ = model(X_train)
    loss = criterion(logits, y_train)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 10 == 0:
        # Compute accuracy
        with torch.no_grad():
            predictions = logits.argmax(dim=1)
            accuracy = (predictions == y_train).float().mean()
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}")

# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Visualize Attention Weights

In [ ]:
# Get attention weights for a sample
model.eval()
sample_idx = 0  # "this movie is great ."

with torch.no_grad():
    sample_input = X_train[sample_idx:sample_idx+1]
    logits, attn_weights_list = model(sample_input)
    prediction = logits.argmax(dim=1)

# Get tokens for visualization
idx_to_word = {v: k for k, v in vocab.items()}
tokens = [idx_to_word[idx.item()] for idx in sample_input[0]]

print(f"Sentence: {' '.join(tokens)}")
print(f"True label: {y_train[sample_idx].item()} (0=negative, 1=positive)")
print(f"Predicted: {prediction.item()} (0=negative, 1=positive)")
print(f"Confidence: {F.softmax(logits, dim=1)[0, prediction].item():.4f}")

# Visualize attention from each layer and head
for layer_idx, attn in enumerate(attn_weights_list):
    # attn shape: [batch, num_heads, seq_len, seq_len]
    attn = attn[0].cpu().numpy()  # Get first batch item
    
    fig, axes = plt.subplots(1, num_heads, figsize=(20, 4))
    fig.suptitle(f'Layer {layer_idx + 1} - Attention Weights', fontsize=14)
    
    for head_idx in range(num_heads):
        ax = axes[head_idx]
        sns.heatmap(attn[head_idx], 
                   xticklabels=tokens, 
                   yticklabels=tokens,
                   cmap='Blues',
                   ax=ax,
                   cbar=True,
                   square=True)
        ax.set_title(f'Head {head_idx + 1}')
        ax.set_xlabel('Key')
        ax.set_ylabel('Query')
    
    plt.tight_layout()
    plt.show()

## 10. Test Different Sentences

In [ ]:
def predict_sentiment(sentence, model, vocab, max_len=10):
    """Predict sentiment of a sentence"""
    model.eval()
    
    # Tokenize and pad
    tokens = tokenize(sentence, vocab)
    padded = pad_sequence([tokens], max_len, vocab['<PAD>'])[0]
    
    # Convert to tensor
    x = torch.tensor([padded]).to(device)
    
    # Predict
    with torch.no_grad():
        logits, attn_weights = model(x)
        probs = F.softmax(logits, dim=1)
        prediction = logits.argmax(dim=1).item()
    
    sentiment = "POSITIVE" if prediction == 1 else "NEGATIVE"
    confidence = probs[0, prediction].item()
    
    print(f"Sentence: {sentence}")
    print(f"Sentiment: {sentiment}")
    print(f"Confidence: {confidence:.4f}")
    print(f"Probabilities - Negative: {probs[0, 0]:.4f}, Positive: {probs[0, 1]:.4f}")
    print()

# Test on training examples
test_sentences = [
    "this movie is great .",
    "this movie is terrible .",
    "love this film .",
    "hate this movie .",
]

for sentence in test_sentences:
    predict_sentiment(sentence, model, vocab)

## 11. Summary and Key Takeaways

### What We Implemented:
1. **Scaled Dot-Product Attention** - The core mechanism
2. **Positional Encoding** - Adding position information
3. **Multi-Head Attention** - Multiple attention heads in parallel
4. **Feed-Forward Network** - Position-wise transformation
5. **Transformer Block** - Complete building block with residual connections
6. **Sentiment Classifier** - End-to-end application

### Key Differences from Bahdanau/Luong:
- **Self-attention**: Q, K, V all from same sequence
- **No RNN**: Fully parallel, no sequential dependencies
- **Positional encoding**: Must explicitly add position info
- **Multi-head**: Multiple attention patterns in parallel
- **Scalable**: O(n²) but fully parallelizable

### Architecture:
```
Input → Embedding + Positional Encoding
     ↓
   [Multi-Head Self-Attention → Add & Norm → FFN → Add & Norm] × N layers
     ↓
   Output (classification, generation, etc.)
```

### Important Formulas:
- **Attention**: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$
- **Multi-Head**: $\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$
- **Positional Encoding**: $PE_{(pos, 2i)} = \sin(pos/10000^{2i/d})$

### Next Steps:
1. Try on a real dataset (IMDB, SST-2)
2. Implement encoder-decoder architecture (for translation)
3. Add more sophisticated positional encodings (learned, rotary)
4. Experiment with different attention variants (sparse, linear)
5. Build a language model (GPT-style)
6. Study BERT, GPT architectures in detail

## 12. Comparison Visualization: Self-Attention vs Cross-Attention

In [ ]:
print("="*60)
print("COMPARISON: Self-Attention vs Cross-Attention (Bahdanau/Luong)")
print("="*60)

comparison = """
┌─────────────────────────────────────────────────────────────┐
│                    CROSS-ATTENTION                          │
│              (Bahdanau/Luong Attention)                     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Encoder: "I love cats"  →  [h1, h2, h3]  (Keys, Values)   │
│                                  ↓                          │
│  Decoder: [s1, s2, ...]      (Queries)                      │
│                                  ↓                          │
│  Decoder attends to Encoder outputs                        │
│                                                             │
│  Use case: Translation, Summarization                       │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                    SELF-ATTENTION                           │
│                  (Transformer)                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Input: "I love cats"  →  [x1, x2, x3]                     │
│                              ↓                              │
│  Q = X @ W_Q  (Queries)   ←─┐                              │
│  K = X @ W_K  (Keys)      ←─┼─ All from same source!       │
│  V = X @ W_V  (Values)    ←─┘                              │
│                              ↓                              │
│  Each position attends to all positions in same sequence   │
│                                                             │
│  Use case: Encoding, Language Modeling, Classification     │
└─────────────────────────────────────────────────────────────┘
"""

print(comparison)